# 0. Settings

In [ ]:
# Optional: install once in Colab
# !pip install -q nnsight

# Optional Drive mount (Colab only)
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

from pathlib import Path

import pandas as pd
import torch
import sys
# from nnsight import LanguageModel

# -----------------------------
# Experiment config
# -----------------------------
BASE_DIR = Path('.')
DATASET_DIR = BASE_DIR / 'dataset'
RESULTS_DIR = BASE_DIR / 'results'
SRC_DIR = BASE_DIR / 'src'

DATASETS = ['antonym.json']
MODEL_NAME = 'meta-llama/Llama-3.2-3B'
RELATIONS = ['antonym', 'none', 'repeat']
INTERVENTION_SOURCE_REL = 'antonym'
INTERVENTION_TARGET_REL = 'none'
INTERVENTION_SAMPLE_LIMIT = 25
PROMPT_TEMPLATE = "Relation: {relation}\nInput: {input}\nOutput: {output}"
ALTERNATIVE_PROMPT_TEMPLATE = "Q: What is the {relation} of {input}?\nA: {output}"

# -----------------------------
# Derived paths
# -----------------------------
import hashlib
DATASET_PATH = DATASET_DIR / DATASETS[0]
SAFE_MODEL_NAME = MODEL_NAME.replace('/', '_')
PROMPT_HASH = hashlib.sha256(PROMPT_TEMPLATE.encode()).hexdigest()[:8]
SAVE_DIR = RESULTS_DIR / SAFE_MODEL_NAME / DATASET_PATH.stem / f'prompt_{PROMPT_HASH}'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MEAN_VECTOR_PATH = SAVE_DIR / 'mean_vector.pt'
SCORE_PATH = SAVE_DIR / 'score.pt'
RELATION_VECTOR_PATH = SAVE_DIR / 'relation_vector.pt'
PATCHING_HEAD_VIZ_PATH = SAVE_DIR / 'patching_head_viz.png'
INTERVENTION_RESULTS_PATH = SAVE_DIR / 'intervention_results.csv'
INTERVENTION_FIG_PATH = SAVE_DIR / 'intervention_fig.png'
INTERVENTION_RESTORE_FIG_PATH = SAVE_DIR / 'intervention_restore_fig.png'
INTERVENTION_ENFORCE_FIG_PATH = SAVE_DIR / 'intervention_enforce_fig.png'

# 1. Loading Utilities

In [ ]:
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import evaluation_helpers
evaluation_helpers.DATASET_DIR = DATASET_DIR

from evaluation_helpers import (
    load_hf_model_and_tokenizer,
    get_first_token_id,
    summarize_layer_metrics,
    run_evaluation,
)

from patching_helpers import (
    set_seed,
    filter_correct_samples,
    convert_relation,
    split_correct_samples,
    cache_activations,
    compute_head_intervention_scores,
    filter_top_heads,
    build_intervention_vector,
)

from intervention_helpers import (
    evaluate_layer_intervention,
)

from visualization_helpers import (
    visualize_multiple_comparisons,
    plot_patching_heatmap,
    plot_intervention_results,
    plot_combined_intervention_results,
    plot_intervention_accuracy_comparison,
    plot_layer_intervention_accuracy,
    visualize_intervention_scores,
    plot_accuracy_barplot,
)

from io_helpers import (
    save_file,
)

# 2. Loading Model

In [ ]:
model, tokenizer = load_hf_model_and_tokenizer(MODEL_NAME)
nn_model = LanguageModel(model)

print(f'Model      : {MODEL_NAME}')
print(f'Dataset    : {DATASET_PATH.name}')
print(f'Save dir   : {SAVE_DIR}')


# 3. Model Evaluation on the Relations

In [ ]:
def run_base_evaluation():
    eval_results = run_evaluation(
        model,
        tokenizer,
        RELATIONS,
        DATASET_PATH,
        SAVE_DIR,
        prompt_template=PROMPT_TEMPLATE,
    )

    base_df = eval_results[INTERVENTION_SOURCE_REL]
    print(f'Base accuracy ({INTERVENTION_SOURCE_REL}): {base_df["output_prediction"].mean():.4f}')
    return eval_results

In [ ]:
eval_results = run_base_evaluation()
base_df = eval_results[INTERVENTION_SOURCE_REL]

# 4. Accuracy Barplot

In [ ]:
eval_dfs = {rel: pd.read_csv(SAVE_DIR / f'_{rel}.csv') for rel in RELATIONS}

plot_accuracy_barplot(eval_dfs, save_path=SAVE_DIR / 'accuracy_plot.png')

# 5. Extracting a Relation Vector

In [ ]:
def run_relation_vector_pipeline(base_df: pd.DataFrame):
    filtered_df = filter_correct_samples(base_df)
    print(f'Correct samples used for vector extraction: {len(filtered_df)}')

    set_seed()
    clean_df, corrupt_df = split_correct_samples(filtered_df, prompt_template=PROMPT_TEMPLATE)

    mean_out = cache_activations(nn_model, clean_df)
    score = compute_head_intervention_scores(nn_model, corrupt_df, mean_out)

    save_file(mean_out, MEAN_VECTOR_PATH)
    save_file(score, SCORE_PATH)

    visualize_intervention_scores(score, save_path=PATCHING_HEAD_VIZ_PATH)

    intervention_vector = build_intervention_vector(
        model,
        filter_top_heads(score),
        mean_out,
    )
    save_file(intervention_vector, RELATION_VECTOR_PATH)

    return {
        'filtered_df': filtered_df,
        'clean_df': clean_df,
        'corrupt_df': corrupt_df,
        'mean_out': mean_out,
        'score': score,
        'intervention_vector': intervention_vector,
    }

In [ ]:
vector_artifacts = run_relation_vector_pipeline(base_df)
filtered_df = vector_artifacts['filtered_df']
intervention_vector = vector_artifacts['intervention_vector']

# 6. Intervention on Sub Datasets

In [ ]:
def run_layer_intervention_experiment(
    df: pd.DataFrame,
    intervention_vector: torch.Tensor,
    source_relation: str,
    target_relation: str,
    save_csv_path: Path | None = None,
    save_fig_path: Path | None = None,
    sample_limit: int | None = INTERVENTION_SAMPLE_LIMIT,
):
    intervention_df = convert_relation(df, target_relation, prompt_template=PROMPT_TEMPLATE)

    if sample_limit is not None:
        intervention_df = intervention_df[:sample_limit]

    results_df = evaluate_layer_intervention(
        nn_model,
        intervention_df,
        intervention_vector,
    )
    # TODO: Save layer_metrics
    layer_metrics = summarize_layer_metrics(results_df)

    if save_csv_path is not None:
        save_file(results_df, save_csv_path)

    if save_fig_path is not None:
        plot_layer_intervention_accuracy(
            layer_metrics,
            model_name=MODEL_NAME,
            save_path=save_fig_path,
        )

    return results_df, layer_metrics

## 6-0. Correct Dataset

In [ ]:
intervention_results_df, layer_metrics = run_layer_intervention_experiment(
    filtered_df,
    intervention_vector,
    source_relation=INTERVENTION_SOURCE_REL,
    target_relation=INTERVENTION_TARGET_REL,
    save_csv_path=INTERVENTION_RESULTS_PATH,
    save_fig_path=INTERVENTION_FIG_PATH,
)

## 6-1. Incorrect Dataset with Perturbation

In [ ]:
incorrect_df = filter_correct_samples(base_df, correct=False)

intervention_out_results_df, layer_out_metrics = run_layer_intervention_experiment(
    incorrect_df,
    intervention_vector,
    source_relation=INTERVENTION_SOURCE_REL,
    target_relation=INTERVENTION_TARGET_REL,
    save_fig_path=INTERVENTION_RESTORE_FIG_PATH,
)

## 6-2. Incorrect Dataset without Perturbation

In [ ]:
intervention_in_df = incorrect_df[:INTERVENTION_SAMPLE_LIMIT]
intervention_in_results_df = evaluate_layer_intervention(
    nn_model,
    intervention_in_df,
    intervention_vector,
)
layer_in_metrics = summarize_layer_metrics(intervention_in_results_df)

plot_layer_intervention_accuracy(
    layer_in_metrics,
    model_name=MODEL_NAME,
    save_path=INTERVENTION_ENFORCE_FIG_PATH,
)